<a href="https://colab.research.google.com/github/robotics-hana/COMP0173_T1_25/blob/main/Preprocessing%20Iraq%20Marshes%20Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [52]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [62]:
!mkdir -p "/content/drive/MyDrive/Iraq_Marshes/images"
!mkdir -p "/content/drive/MyDrive/Iraq_Marshes/masks"
!mv "/content/drive/MyDrive/Iraq_Marshes/"*"_s2_4band"*.tif* "/content/drive/MyDrive/Iraq_Marshes/images/" 2>/dev/null || true
!mv "/content/drive/MyDrive/Iraq_Marshes/"*"_gsw_water"*.tif* "/content/drive/MyDrive/Iraq_Marshes/masks/" 2>/dev/null || true

!echo "IMAGES:"; ls "/content/drive/MyDrive/Iraq_Marshes/images" | head -n 20
!echo "MASKS:";  ls "/content/drive/MyDrive/Iraq_Marshes/masks"  | head -n 20

image_dir = "/content/drive/MyDrive/Iraq_Marshes/images"
mask_dir  = "/content/drive/MyDrive/Iraq_Marshes/masks"
!mkdir -p "/content/drive/MyDrive/Iraq_Marshes/Iraq_Marshes_s2_water_seg"


IMAGES:
central_marshes_2017_s2_4band_8bit.tif
central_marshes_2021_s2_4band_8bit.tif
MASKS:
central_marshes_2017_gsw_water.tif
central_marshes_2021_gsw_water.tif


In [66]:
# ====== Iraq Marshes dataset: tile + preprocessing + balanced subsample + save .npy ======
import os, glob, random
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

# ---------------------------
# 0) PATHS + SETTINGS
# ---------------------------
GEE_IMG_DIR = "/content/drive/MyDrive/Iraq_Marshes/images"
GEE_MSK_DIR = "/content/drive/MyDrive/Iraq_Marshes/masks"
OUT_ROOT    = "/content/drive/MyDrive/Iraq_Marshes/Iraq_Marshes_s2_water_seg"

TILE = 512
SEED = 42

# Target counts (2021 only)
TARGET_TRAIN = 250
TARGET_VAL   = 100
TARGET_TEST  = 20

# Water coverage buckets
WATER_HEAVY_FRAC = 0.30  # 30%+ of pixels are water => "water-heavy"

# Bucket proportions within each split (must sum to 1.0)
P_EMPTY = 0.33
P_MIXED = 0.34
P_HEAVY = 0.33

random.seed(SEED)
np.random.seed(SEED)

# ---------------------------
# 1) HELPERS
# ---------------------------
def make_dirs(root: str) -> None:
    for split in ["training", "validation", "test"]:
        os.makedirs(os.path.join(root, split, "images"), exist_ok=True)
        os.makedirs(os.path.join(root, split, "masks"), exist_ok=True)

def align_mask_to_image(img_path: str, mask_path: str):
    """
    Returns:
      img_arr: (H, W, 4) float32
      mask_arr: (H, W) uint8 in {0,1}
    """
    with rasterio.open(img_path) as src_img:
        img = src_img.read().astype(np.float32)   # (bands,H,W)
        img_transform = src_img.transform
        img_crs = src_img.crs
        img_h, img_w = src_img.height, src_img.width

    img = np.nan_to_num(img, nan=0.0, posinf=0.0, neginf=0.0)

    with rasterio.open(mask_path) as src_msk:
        msk = src_msk.read(1).astype(np.float32)
        msk = np.nan_to_num(msk, nan=0.0, posinf=0.0, neginf=0.0)

        same_grid = (
            src_msk.crs == img_crs and
            src_msk.transform == img_transform and
            src_msk.width == img_w and
            src_msk.height == img_h
        )

        if same_grid:
            mask_resampled = msk
        else:
            mask_resampled = np.zeros((img_h, img_w), dtype=np.float32)
            reproject(
                source=msk,
                destination=mask_resampled,
                src_transform=src_msk.transform,
                src_crs=src_msk.crs,
                dst_transform=img_transform,
                dst_crs=img_crs,
                resampling=Resampling.nearest
            )

    img_arr = np.transpose(img, (1, 2, 0))             # (H,W,4)
    mask_arr = (mask_resampled > 0.5).astype(np.uint8) # (H,W) 0/1
    return img_arr, mask_arr

def per_tile_minmax(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    mn = float(np.min(x))
    mx = float(np.max(x))
    if mx <= mn:
        return np.zeros_like(x, dtype=np.float32)
    return (x - mn) / (mx - mn)

def water_fraction(msk_t: np.ndarray) -> float:
    return float(msk_t.mean())

def parse_base_and_year(s2_path: str):
    """
    central_marshes_2017_s2_4band_8bit.tif -> base='central_marshes_2017', year=2017
    """
    fname = os.path.basename(s2_path)
    stem = os.path.splitext(fname)[0]
    base = stem.replace("_s2_4band_8bit", "")
    year = int(base.split("_")[-1])
    return base, year

def collect_tiles():
    s2_files = sorted(glob.glob(os.path.join(GEE_IMG_DIR, "*_s2_4band_8bit.tif*")))
    if not s2_files:
        raise FileNotFoundError(f"No S2 files found in: {GEE_IMG_DIR}")

    tiles = []
    for s2_path in s2_files:
        base, year = parse_base_and_year(s2_path)

        # ✅ 2021 ONLY
        if year != 2021:
            continue

        msk_path = os.path.join(GEE_MSK_DIR, base + "_gsw_water.tif")
        if not os.path.exists(msk_path):
            msk_path = os.path.join(GEE_MSK_DIR, base + "_gsw_water.tiff")

        if not os.path.exists(msk_path):
            print(f"⚠ Missing mask for {base}, skipping")
            continue

        print("Reading + aligning:", base)
        img_arr, mask_arr = align_mask_to_image(s2_path, msk_path)

        H, W, C = img_arr.shape
        if C != 4:
            print(f"⚠ Expected 4 bands, got {C} for {base}. Continuing anyway.")

        for r0 in range(0, H - TILE + 1, TILE):
            for c0 in range(0, W - TILE + 1, TILE):
                img_t = img_arr[r0:r0+TILE, c0:c0+TILE, :]
                msk_t = mask_arr[r0:r0+TILE, c0:c0+TILE]

                wf = water_fraction(msk_t)
                tile_id = f"{base}_r{r0//TILE:03d}_c{c0//TILE:03d}"

                tiles.append({
                    "id": tile_id,
                    "img": img_t,
                    "msk": msk_t,
                    "year": year,
                    "wf": wf
                })

    if not tiles:
        raise RuntimeError("No 2021 tiles created. Check your 2021 rasters and masks.")
    return tiles

def bucket_tiles(tiles):
    empty = [t for t in tiles if t["wf"] == 0.0]
    heavy = [t for t in tiles if t["wf"] >= WATER_HEAVY_FRAC]
    mixed = [t for t in tiles if (t["wf"] > 0.0 and t["wf"] < WATER_HEAVY_FRAC)]
    return empty, mixed, heavy

def sample_k(lst, k):
    if k <= 0:
        return []
    k = min(k, len(lst))
    random.shuffle(lst)
    return lst[:k]

def build_split_balanced(tiles_pool, n_total):
    """
    Pick a balanced set of empty/mixed/heavy tiles from a pool.
    Tops up from remaining tiles if a bucket is too small.
    """
    empty, mixed, heavy = bucket_tiles(tiles_pool)

    n_e = int(round(P_EMPTY * n_total))
    n_m = int(round(P_MIXED * n_total))
    n_h = n_total - n_e - n_m

    chosen = []
    chosen += sample_k(empty.copy(), n_e)
    chosen += sample_k(mixed.copy(), n_m)
    chosen += sample_k(heavy.copy(), n_h)

    if len(chosen) < n_total:
        chosen_ids = set(t["id"] for t in chosen)
        remaining = [t for t in tiles_pool if t["id"] not in chosen_ids]
        chosen += sample_k(remaining, n_total - len(chosen))

    random.shuffle(chosen)
    return chosen

def split_2021_stratified_nonoverlap(tiles_2021, n_train, n_val, n_test, min_heavy_val, min_heavy_test):
    """
    Ensures val/test get at least some heavy tiles (if available),
    then fills the rest using balanced sampling from remaining tiles.
    """

    empty, mixed, heavy = bucket_tiles(tiles_2021)
    random.shuffle(empty); random.shuffle(mixed); random.shuffle(heavy)

    # take heavy for val/test FIRST
    hv = min(min_heavy_val, len(heavy))
    val_heavy = heavy[:hv]
    heavy = heavy[hv:]

    ht = min(min_heavy_test, len(heavy))
    test_heavy = heavy[:ht]
    heavy = heavy[ht:]

    # remaining pool after reserving
    remaining = empty + mixed + heavy
    random.shuffle(remaining)

    def build_from_pool(pool, n_total):
        empty_p, mixed_p, heavy_p = bucket_tiles(pool)

        # use your target proportions
        n_e = int(round(P_EMPTY * n_total))
        n_m = int(round(P_MIXED * n_total))
        n_h = n_total - n_e - n_m

        chosen = []
        chosen += sample_k(empty_p.copy(), n_e)
        chosen += sample_k(mixed_p.copy(), n_m)
        chosen += sample_k(heavy_p.copy(), n_h)

        if len(chosen) < n_total:
            chosen_ids = set(t["id"] for t in chosen)
            rem = [t for t in pool if t["id"] not in chosen_ids]
            chosen += sample_k(rem, n_total - len(chosen))

        return chosen

    # TRAIN from all remaining (no overlap tracking needed yet)
    train = build_from_pool(remaining, n_train)
    used = set(t["id"] for t in train)
    rem_after_train = [t for t in remaining if t["id"] not in used]

    # VAL: start with reserved heavy then fill remainder
    need_val = n_val - len(val_heavy)
    val_fill = build_from_pool(rem_after_train, need_val)
    val = val_heavy + val_fill
    used |= set(t["id"] for t in val_fill)
    rem_after_val = [t for t in rem_after_train if t["id"] not in used]

    # TEST: start with reserved heavy then fill remainder
    need_test = n_test - len(test_heavy)
    test_fill = build_from_pool(rem_after_val, need_test)
    test = test_heavy + test_fill

    random.shuffle(train); random.shuffle(val); random.shuffle(test)
    return train, val, test

def save_as_npy(tiles_list, split_name):
    img_out = os.path.join(OUT_ROOT, split_name, "images")
    msk_out = os.path.join(OUT_ROOT, split_name, "masks")

    for t in tiles_list:
        x = per_tile_minmax(t["img"]).astype(np.float32)  # (512,512,4)
        y = t["msk"].astype(np.uint8)[..., None]          # (512,512,1)
        np.save(os.path.join(img_out, t["id"] + ".npy"), x)
        np.save(os.path.join(msk_out, t["id"] + ".npy"), y)

def summarise_split(name, tiles_list):
    wfs = np.array([t["wf"] for t in tiles_list], dtype=np.float32)
    pct_nonempty = float((wfs > 0).mean() * 100.0)
    pct_heavy = float((wfs >= WATER_HEAVY_FRAC).mean() * 100.0)
    print(f"{name}: n={len(tiles_list)} | non-empty={pct_nonempty:.1f}% | heavy(>={WATER_HEAVY_FRAC:.2f})={pct_heavy:.1f}% | wf_mean={wfs.mean():.3f}")

# ---------------------------
# 2) RUN
# ---------------------------
make_dirs(OUT_ROOT)

tiles_2021 = collect_tiles()
print("Total 2021 tiles created:", len(tiles_2021))

train_tiles, val_tiles, test_tiles = split_2021_stratified_nonoverlap(
    tiles_2021,
    TARGET_TRAIN,
    TARGET_VAL,
    TARGET_TEST,
    min_heavy_val=10,
    min_heavy_test=3
)

print("Final Train/Val/Test:", len(train_tiles), len(val_tiles), len(test_tiles))
summarise_split("TRAIN", train_tiles)
summarise_split("VAL  ", val_tiles)
summarise_split("TEST ", test_tiles)

save_as_npy(train_tiles, "training")
save_as_npy(val_tiles, "validation")
save_as_npy(test_tiles, "test")

print("✅ Done. Saved to:", OUT_ROOT)


Reading + aligning: central_marshes_2021
Total 2021 tiles created: 1116
Final Train/Val/Test: 250 100 20
TRAIN: n=250 | non-empty=62.0% | heavy(>=0.30)=22.0% | wf_mean=0.158
VAL  : n=100 | non-empty=55.0% | heavy(>=0.30)=10.0% | wf_mean=0.095
TEST : n=20 | non-empty=45.0% | heavy(>=0.30)=15.0% | wf_mean=0.075
✅ Done. Saved to: /content/drive/MyDrive/Iraq_Marshes/Iraq_Marshes_s2_water_seg


In [67]:
!ls -lah "/content/drive/MyDrive/Iraq_Marshes/Iraq_Marshes_s2_water_seg"
!ls -lah "/content/drive/MyDrive/Iraq_Marshes/Iraq_Marshes_s2_water_seg/validation/images" | head
!ls -lah "/content/drive/MyDrive/Iraq_Marshes/Iraq_Marshes_s2_water_seg/test/images" | head


total 12K
drwx------ 4 root root 4.0K Jan 17 16:27 test
drwx------ 4 root root 4.0K Jan 17 16:27 training
drwx------ 4 root root 4.0K Jan 17 16:27 validation
total 401M
-rw------- 1 root root 4.1M Jan 17 16:31 central_marshes_2021_r000_c002.npy
-rw------- 1 root root 4.1M Jan 17 16:31 central_marshes_2021_r001_c000.npy
-rw------- 1 root root 4.1M Jan 17 16:31 central_marshes_2021_r001_c008.npy
-rw------- 1 root root 4.1M Jan 17 16:31 central_marshes_2021_r001_c011.npy
-rw------- 1 root root 4.1M Jan 17 16:31 central_marshes_2021_r001_c021.npy
-rw------- 1 root root 4.1M Jan 17 16:31 central_marshes_2021_r001_c024.npy
-rw------- 1 root root 4.1M Jan 17 16:31 central_marshes_2021_r002_c032.npy
-rw------- 1 root root 4.1M Jan 17 16:31 central_marshes_2021_r002_c034.npy
-rw------- 1 root root 4.1M Jan 17 16:31 central_marshes_2021_r002_c035.npy
total 81M
-rw------- 1 root root 4.1M Jan 17 16:31 central_marshes_2021_r000_c018.npy
-rw------- 1 root root 4.1M Jan 17 16:31 central_marshes_2021